# 10. Stable Diffusion 기반 이미지 생성

Kaggle Notebook에서 바로 실행할 수 있는 Stable Diffusion 이미지 생성 예제입니다.

- `diffusers` 라이브러리 사용
- `StableDiffusionPipeline` 사용
- 텍스트 프롬프트를 입력으로 이미지 생성
- `num_inference_steps`, `guidance_scale`, `height`, `width` 옵션 포함
- 생성된 이미지를 화면에 출력하고 파일로 저장

> Kaggle에서 모델 다운로드가 필요하므로 Notebook 오른쪽 설정에서 Internet을 켜 주세요. GPU 사용을 권장합니다.

In [ ]:
# Kaggle 환경에서 필요한 패키지를 설치합니다.
# 이미 설치되어 있으면 빠르게 넘어갑니다.
!pip -q install diffusers transformers accelerate safetensors

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

# GPU가 있으면 cuda를 사용하고, 없으면 cpu를 사용합니다.
# Stable Diffusion은 CPU에서도 실행은 가능하지만 매우 느립니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

print("사용 장치:", device)
print("torch dtype:", torch_dtype)

## 1. Stable Diffusion Pipeline 불러오기

In [ ]:
# Stable Diffusion v1.5 모델을 사용합니다.
# 처음 실행할 때는 Hugging Face에서 모델 파일을 다운로드하므로 시간이 걸릴 수 있습니다.
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    use_safetensors=True,
)
pipe = pipe.to(device)

# GPU 메모리 사용량을 줄이기 위한 설정입니다.
pipe.enable_attention_slicing()

print("Stable Diffusion Pipeline 로딩 완료:", model_id)

## 2. 프롬프트와 이미지 생성 옵션 설정

`prompt`에 만들고 싶은 이미지를 영어로 입력합니다. Stable Diffusion은 영어 프롬프트에서 보통 더 안정적으로 동작합니다.

In [ ]:
# 이미지 생성을 위한 텍스트 프롬프트입니다.
prompt = "a cozy futuristic library with warm lights, detailed digital art"

# 원하지 않는 요소를 줄이고 싶을 때 사용하는 네거티브 프롬프트입니다.
negative_prompt = "low quality, blurry, distorted, bad anatomy, text, watermark"

# 이미지 생성 옵션입니다.
# num_inference_steps: 높을수록 품질이 좋아질 수 있지만 생성 시간이 늘어납니다.
# guidance_scale: 높을수록 프롬프트를 더 강하게 따르지만 너무 높으면 부자연스러울 수 있습니다.
# height, width: 8의 배수로 지정해야 하며, Kaggle GPU에서는 512x512부터 권장합니다.
num_inference_steps = 30
guidance_scale = 7.5
height = 512
width = 512

# 같은 프롬프트에서 결과를 재현하고 싶으면 seed를 고정합니다.
seed = 42
generator = torch.Generator(device=device).manual_seed(seed) if device == "cuda" else torch.Generator().manual_seed(seed)

print("Prompt:", prompt)
print("Negative prompt:", negative_prompt)
print("옵션:", num_inference_steps, guidance_scale, height, width)

## 3. 프롬프트 기반 이미지 생성하기

In [ ]:
# StableDiffusionPipeline에 프롬프트와 옵션을 전달하여 이미지를 생성합니다.
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    height=height,
    width=width,
    generator=generator,
)

# 생성 결과에서 첫 번째 이미지를 꺼냅니다.
image = result.images[0]

# Notebook 화면에 이미지를 출력합니다.
display(image)

## 4. 생성된 이미지를 파일로 저장하기

In [ ]:
# Kaggle에서는 /kaggle/working 폴더에 저장하면 Output에서 다운로드할 수 있습니다.
output_dir = Path("/kaggle/working/stable_diffusion_outputs")

# 로컬 Jupyter에서 실행하는 경우 /kaggle/working이 없을 수 있으므로 현재 폴더에 저장합니다.
if not Path("/kaggle/working").exists():
    output_dir = Path("stable_diffusion_outputs")

output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"stable_diffusion_{timestamp}.png"

image.save(output_path)

print("저장 완료:", output_path)

## 5. 여러 프롬프트로 이미지 여러 장 생성하기

아래 셀은 여러 개의 프롬프트를 한 번에 실행하고 각각 파일로 저장하는 예시입니다.

In [ ]:
prompts = [
    "a small robot reading a book in a sunlit room, cute digital art",
    "a peaceful mountain lake at sunrise, cinematic landscape photography",
    "a modern city street in the rain at night, neon reflections, realistic",
]

generated_files = []

for index, prompt_text in enumerate(prompts, start=1):
    print(f"\n[{index}] 이미지 생성 중:", prompt_text)

    # 프롬프트마다 다른 seed를 사용합니다.
    current_seed = seed + index
    current_generator = (
        torch.Generator(device=device).manual_seed(current_seed)
        if device == "cuda"
        else torch.Generator().manual_seed(current_seed)
    )

    current_result = pipe(
        prompt=prompt_text,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        height=height,
        width=width,
        generator=current_generator,
    )

    current_image = current_result.images[0]
    display(current_image)

    current_path = output_dir / f"stable_diffusion_{index:02d}_seed_{current_seed}.png"
    current_image.save(current_path)
    generated_files.append(str(current_path))

    print("저장 완료:", current_path)

generated_files